# M2 Notebook 20 — Interpretable Machine Learning

**Status:** Runnable first edition

## Learning objectives

- Compute permutation importance.
- Create partial-dependence profiles.
- Build local linear explanations.
- Recognize interpretability limitations.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    LinearRegression,RandomForestRegressor,feature_effect_correlation,
    local_linear_explanation,mean_squared_error,partial_dependence,
    permutation_importance,train_test_split,
)


In [ ]:
rng=np.random.default_rng(20)
X=rng.normal(size=(600,4))
y=2*X[:,0]+0.5*X[:,1]**2+rng.normal(scale=.5,size=600)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,seed=20)
model=RandomForestRegressor(80,max_depth=6,seed=20).fit(Xtr,ytr)
importance=permutation_importance(
    model,Xte,yte,mean_squared_error,n_repeats=5,seed=20,higher_is_better=False
)
pd.Series(importance,index=[f"Feature_{i+1}" for i in range(4)]).sort_values(ascending=False)


## Partial dependence

In [ ]:
grid=np.linspace(np.quantile(Xte[:,0],.05),np.quantile(Xte[:,0],.95),60)
pd_values=partial_dependence(model,Xte,0,grid)
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(grid,pd_values)
ax.set_xlabel("Feature 1"); ax.set_ylabel("Average prediction")
ax.set_title("Partial Dependence")
plt.show()


## Local explanation

In [ ]:
explanation=local_linear_explanation(model,Xte[0],scale=.2,samples=1000,seed=20)
explanation


## Interpretability cautions

Feature importance is model-specific, correlated features can share or mask importance, and local explanations may not be globally valid.

## Decision Intelligence case

Explanations support review and challenge, but should not be presented as causal proof.

## Key insight

Interpretability tools explain model behavior under particular assumptions; they do not automatically explain the real world.